# Reward Utils Demo Notebook

This notebook is a standalone walkthrough of the `reward_utils` pipeline.

It demonstrates, in a simple and executable way:

- molecule parsing from SMILES and SELFIES
- canonicalization and cross-representation comparison
- Dice and Tanimoto similarity on Morgan fingerprints
- `rmatch`, `rdiv`, and combined reward
- sequence scoring with valid, duplicate, and invalid molecules

Every code block ends with explicit validation prints so you can see what is happening at each stage.

## 1. Setup

This block makes the notebook robust to being executed from inside `reward_utils/` or from the repo root.

It imports the current module implementation directly, so the notebook stays aligned with the tested code.

In [ ]:
from pathlib import Path
import sys

cwd = Path.cwd().resolve()
project_root = cwd.parent if cwd.name == 'reward_utils' else cwd
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import selfies
from reward_utils import (
    DEFAULT_REWARD_CONFIG,
    CHEBI20_REWARD_CONFIG,
    compute_dice_similarity,
    compute_rdiv,
    compute_rmatch,
    compute_tanimoto_similarity,
    compute_total_reward,
    parse_molecule_text,
    score_candidate_sequence,
)
import rdkit
rdkit_version = rdkit.__version__

print('Current working directory:', cwd)
print('Resolved project root:', project_root)
print('selfies version:', selfies.__version__)
print('rdkit version:', rdkit_version)
print('Default reward config:', DEFAULT_REWARD_CONFIG)
print('ChEBI-20 reward config:', CHEBI20_REWARD_CONFIG)

## 2. Parse Individual Molecules

The parser accepts SMILES, SELFIES, and `representation='auto'`.

This is the first place to validate that the reward pipeline can normalize different representations into the same canonical structure.

In [ ]:
def summarize_record(record):
    return {
        'input_text': record.input_text,
        'representation': record.input_representation,
        'normalized_input': record.normalized_input,
        'canonical_smiles': record.canonical_smiles,
        'is_valid': record.is_valid,
        'used_selfies_decoder': record.used_selfies_decoder,
        'error': record.error,
    }

records = {
    'smiles_ethanol': parse_molecule_text('CCO', representation='smiles'),
    'selfies_ethanol': parse_molecule_text('[C][C][O]', representation='selfies'),
    'spaced_selfies': parse_molecule_text('[C] [C] [O]', representation='auto'),
    'invalid_example': parse_molecule_text('not a molecule', representation='auto'),
}

for name, record in records.items():
    print(f'--- {name} ---')
    print(summarize_record(record))

## 3. Cross-Representation Equivalence

A core requirement for `reward_utils` is that the same molecule written in SMILES and SELFIES becomes the same canonical molecule after parsing.

This is what makes duplicate checks and reward comparison reliable.

In [ ]:
smiles_record = parse_molecule_text('CCO', representation='smiles')
selfies_record = parse_molecule_text('[C][C][O]', representation='selfies')

same_canonical = smiles_record.canonical_smiles == selfies_record.canonical_smiles

print('SMILES canonical:', smiles_record.canonical_smiles)
print('SELFIES canonical:', selfies_record.canonical_smiles)
print('Same canonical molecule:', same_canonical)
print('Interpretation: SMILES and SELFIES can now be compared in the reward pipeline as the same structure.')

## 4. Similarity Metrics

The paper uses Morgan fingerprints with:

- Dice similarity for `rmatch`
- Tanimoto similarity for `rdiv`

Here we compare the same molecule across representations, then compare similar and less-similar molecules.

In [ ]:
similarity_examples = [
    ('same molecule across rep', '[C][C][O]', 'CCO', 'selfies', 'smiles'),
    ('ethanol vs propane', 'CCO', 'CCC', 'smiles', 'smiles'),
    ('ethanol vs benzene', 'CCO', 'c1ccccc1', 'smiles', 'smiles'),
]

for label, mol_a, mol_b, rep_a, rep_b in similarity_examples:
    dice = compute_dice_similarity(mol_a, mol_b, representation_a=rep_a, representation_b=rep_b)
    tanimoto = compute_tanimoto_similarity(mol_a, mol_b, representation_a=rep_a, representation_b=rep_b)
    print(f'--- {label} ---')
    print('Dice similarity    :', round(dice, 6))
    print('Tanimoto similarity:', round(tanimoto, 6))

print('Validation: same molecule across SMILES/SELFIES should be 1.0 for both metrics.')

## 5. `rmatch`

The description-matching reward is:

`rmatch = max Dice(target, candidate)^alpha`

This cell demonstrates a SELFIES candidate compared against SMILES targets.

In [ ]:
rmatch_component = compute_rmatch(
    '[C][C][O]',
    ['CCC', 'CCO'],
    alpha=0.5,
    candidate_representation='selfies',
    target_representation='smiles',
)

print('rmatch max_similarity:', rmatch_component.max_similarity)
print('rmatch reward        :', rmatch_component.reward)
print('best target index    :', rmatch_component.best_index)
print('best target molecule :', rmatch_component.best_reference)
print('Validation: ethanol written as SELFIES should match target CCO with reward 1.0.')

## 6. `rdiv`

The diversity reward is:

`rdiv = 1 - max Tanimoto(previous, candidate)^beta`

It behaves differently for:

- the first generated candidate
- duplicates
- non-duplicate candidates

In [ ]:
first_step = compute_rdiv('CCO', [], beta=1.0)
duplicate_step = compute_rdiv(
    '[C][C][O]',
    ['CCO', 'CCC'],
    beta=2.0,
    candidate_representation='selfies',
    previous_representation='smiles',
)
novel_step = compute_rdiv('c1ccccc1', ['CCO', 'CCC'], beta=2.0)

print('First candidate rdiv    :', first_step.reward)
print('Duplicate candidate rdiv:', duplicate_step.reward)
print('Novel candidate rdiv    :', novel_step.reward)
print('Validation: first step should be 0.0, duplicate should collapse toward 0.0, novel should stay higher.')

## 7. Combined Reward

The module combines `rmatch` and `rdiv`, then applies the paper-style reward amplification.

This cell compares the same candidate under the default config and the ChEBI-20 config.

In [ ]:
default_breakdown = compute_total_reward(
    '[C][C][O]',
    targets=['CCO', 'CCC'],
    previous_candidates=['CCC'],
    config=DEFAULT_REWARD_CONFIG,
    candidate_representation='selfies',
    target_representation='smiles',
    previous_representation='smiles',
)

chebi_breakdown = compute_total_reward(
    '[C][C][O]',
    targets=['CCO', 'CCC'],
    previous_candidates=['CCC'],
    config=CHEBI20_REWARD_CONFIG,
    candidate_representation='selfies',
    target_representation='smiles',
    previous_representation='smiles',
)

for label, breakdown in [('default', default_breakdown), ('chebi20', chebi_breakdown)]:
    print(f'--- {label} ---')
    print('candidate canonical :', breakdown.candidate.canonical_smiles)
    print('match reward        :', breakdown.match.reward)
    print('diversity reward    :', breakdown.diversity.reward)
    print('total reward        :', breakdown.total_reward)
    print('amplified reward    :', breakdown.amplified_reward)
    print('is duplicate        :', breakdown.is_duplicate)

print('Validation: the ChEBI-20 config changes beta but preserves the same overall pipeline structure.')

## 8. Sequence Scoring

This is the closest standalone demo to how a generation pipeline would consume the reward logic.

Each candidate is scored in order, so the diversity reward depends on earlier molecules.

In [ ]:
sequence_results = score_candidate_sequence(
    candidates=['[C][C][O]', 'CCO', 'not a molecule', 'c1ccccc1'],
    targets=['CCO', 'CCC'],
    config=CHEBI20_REWARD_CONFIG,
    candidate_representation='auto',
    target_representation='smiles',
)

for index, result in enumerate(sequence_results, start=1):
    print(f'--- candidate step {index} ---')
    print('input text         :', result.candidate.input_text)
    print('canonical smiles   :', result.candidate.canonical_smiles)
    print('valid              :', result.candidate.is_valid)
    print('match reward       :', result.match.reward)
    print('diversity reward   :', result.diversity.reward)
    print('total reward       :', result.total_reward)
    print('amplified reward   :', result.amplified_reward)
    print('duplicate          :', result.is_duplicate)
    print('---')

print('Validation: duplicate and invalid molecules should be easy to spot in the step-by-step summary.')

## 9. Curated Reproducibility Example

This mirrors the reproducibility-style regression test in the suite.

Running the same curated sequence twice should produce the same canonical outputs and the same rewards.

In [ ]:
curated_candidates = ['[C][C][O]', 'CCC', 'c1ccccc1', 'CCO']
curated_targets = ['CCO', 'CCC']

run_one = score_candidate_sequence(
    curated_candidates,
    curated_targets,
    config=CHEBI20_REWARD_CONFIG,
    candidate_representation='auto',
    target_representation='smiles',
)
run_two = score_candidate_sequence(
    curated_candidates,
    curated_targets,
    config=CHEBI20_REWARD_CONFIG,
    candidate_representation='auto',
    target_representation='smiles',
)

canonical_equal = [item.candidate.canonical_smiles for item in run_one] == [item.candidate.canonical_smiles for item in run_two]
reward_equal = [item.total_reward for item in run_one] == [item.total_reward for item in run_two]

print('Canonical outputs equal across runs:', canonical_equal)
print('Total rewards equal across runs    :', reward_equal)
print('Run one totals:', [item.total_reward for item in run_one])
print('Run two totals:', [item.total_reward for item in run_two])
print('Validation: deterministic reward behavior is important before PPO integration.')

## 10. Takeaways

This notebook shows that the standalone reward pipeline already supports:

- parsing SMILES and SELFIES
- canonicalizing them into comparable structures
- computing Dice and Tanimoto similarity on Morgan fingerprints
- evaluating `rmatch`, `rdiv`, and the combined reward
- scoring candidate sequences with duplicates and invalid molecules

What it does **not** do yet:

- run PPO or any policy optimization loop
- read prompt-target sets from dataset files
- batch over large evaluation corpora

The intended next step is to connect this verified reward logic into a small PPO or sampling experiment once the reward semantics are fully understood.